In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. SETUP PATHS ---
MAIN_DATA_FILE = 'C:/Users/511232/Desktop/DSS/MERGING GOOGLESHEETS QUESTIONNAIRES/codes/arabic_questionnaires.xlsx'
CRITERIA_FILE = 'C:/Users/511232/Desktop/criterias.xlsx'
OUTPUT_FILE = 'availability_results.xlsx'

# --- 2. LOAD DATA ---
main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE)

# --- 3. PREPARE CRITERIA DICTIONARY ---
# Drop rows where there is no Arabic indicator name to prevent errors
criteria_df = criteria_df.dropna(subset=['Indicator_Ar'])
# Create a dictionary for fast lookup: {'Indicator Name': integer_criteria}
criteria_dict = pd.Series(criteria_df.criteria.values, index=criteria_df['Indicator_Ar']).to_dict()

In [ ]:
# --- 4. DEFINE TIME WINDOWS ---
min_year = 2010
max_year = main_df['السنة'].max()

# range(start, stop, step) creates the edges of our 5-year buckets.
# We use max_year + 6 to ensure the very last year (e.g., 2024 or 2025) is included 
# inside a bin rather than being cut off at the edge.
bins = range(min_year, int(max_year) + 6, 5)

# If bins are [2010, 2015, 2020, 2025], there are 4 edges but only 3 intervals: 
# (2010-14, 2015-19, 2020-24). len(bins) - 1 gives us that count of 3.
all_possible_windows = len(bins) - 1

In [ ]:
# --- 5. DATA CLEANING & SUBSETTING ---
# Define text values that represent "No Data" in the Arabic questionnaires
invalid_entries = ['Not applicable', 'غير متاح', 'Total']

# Identify which rows have valid information for 'Nationality' and 'Area'
# This allows us to calculate availability for specific sub-categories later.
nationality_mask = main_df['المواطنة'].notna() & ~main_df['المواطنة'].isin(invalid_entries)
area_mask = main_df['المنطقة'].notna() & ~main_df['المنطقة'].isin(invalid_entries)

In [ ]:
# --- 6. CORE CALCULATION LOGIC ---
def calculate_metric(target_df):
    """
    Groups data by Indicator/Country and checks if every 5-year window 
    has enough data points based on the criteria file.
    """

    results = []
    # Loop through each unique combination of Indicator and Country
    for (indicator, country), group in target_df.groupby(['المؤشر', 'الدولة']):
        # Look up the required data points for this specific indicator (default to 1)
        criteria = criteria_dict.get(indicator, 1)
        
        # pd.cut assigns every row's year into one of the 5-year bins defined above
        #right=False , open at the tight bracket, to capture 5 years not 6 years
        binned_years = pd.cut(group['السنة'], bins=bins, right=False)
        
        # Count how many data points exist in each 5-year window
        window_counts = binned_years.value_counts()
        
        # Compare every window's count against the criteria. 
        # .sum() here counts how many windows met the requirement.
        sufficient_windows = (window_counts >= criteria).sum()
        
        # If the number of successful windows matches the total possible windows, 
        # mark it as 1 (Available), otherwise 0.
        if sufficient_windows == all_possible_windows:
            is_available = 1
        else:
            is_available = 0
        
        results.append({'المؤشر': indicator, 'الدولة': country, 'status': is_available})
    
    return pd.DataFrame(results)

In [ ]:
# --- 7. RUN CALCULATIONS ---
# Calculate the 3 types of availability using the logic defined above
print("Calculating general availability...")
df_gen = calculate_metric(main_df).rename(columns={'status': 'general_availability'})

print("Calculating nationality availability...")
df_nat = calculate_metric(main_df[nationality_mask]).rename(columns={'status': 'nationality_availability'})

print("Calculating area availability...")
df_area = calculate_metric(main_df[area_mask]).rename(columns={'status': 'area_availability'})

In [ ]:
# --- 8. MERGE RESULTS & EXPORT ---
# Combine the separate dataframes into one master table based on Indicator and Country
final_df = pd.merge(df_gen, df_nat, on=['المؤشر', 'الدولة'], how='left')
final_df = pd.merge(final_df, df_area, on=['المؤشر', 'الدولة'], how='left')

# If a country/indicator had no data at all for a category, merge will create a NaN.
# We fill these with 0 (Not Available) and convert to clean integers.
final_df = final_df.fillna(0)
for col in ['general_availability', 'nationality_availability', 'area_availability']:
    final_df[col] = final_df[col].astype(int)

# Export the final report to Excel
final_df.to_excel(OUTPUT_FILE, index=False)
print(f"Done! Results saved to {OUTPUT_FILE}")